# SoleSearch AI — Step 5: More sneakers, more photos, retrain

This notebook combines **three public datasets** to train on as many sneaker models as possible:

| Source | Size | Labels |
|---|---|---|
| Kaggle "Popular Sneakers Classification" | ~5,000 photos | ~50 sneaker types |
| Kaggle "50k+ Sneaker Images" | ~53,000 photos | 850+ StockX listing names |
| Hugging Face "ipogorelov/sneakers" | ~93,000 photos | manufacturer + model |

Every photo is sorted into one of **50 sneaker models** by its label (Jordan 1–13, Dunks, Air Max, Yeezys, Sambas,
New Balance, ASICS, Converse, Vans, and more). Models with too few photos are skipped automatically.
Duplicates across datasets are removed, and each model is capped so no shoe dominates.

**Everything happens inside Colab.** Nothing is downloaded to your laptop until the very end (one ~20 MB zip).

**Before you start:** Runtime → Change runtime type → **T4 GPU**. The whole notebook takes about **60–90 minutes**.
Keep this tab open while it runs. Your current model is backed up in Drive before anything is replaced.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, random, shutil, time, hashlib, io, os, re
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
from PIL import Image, ImageOps

PROJECT = Path('/content/drive/MyDrive/SoleSearch')
MODEL_DIR = PROJECT / 'models'
MODEL_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_DATA = Path('/content/train_v2')     # built fresh on Colab's fast local disk

MIN_PER_CLASS = 80    # skip sneaker models with fewer photos than this
MAX_PER_CLASS = 600   # cap per model, for balance and training time
IMG_SIDE = 320        # stored size; training uses 224 crops

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: no GPU. Runtime > Change runtime type > T4 GPU, then re-run.')

## 2. The 50 sneaker models and how labels are matched

Each label (a folder name or the Hugging Face text label) is checked against these rules **in order**, and the
first match wins. So "Nike SB Dunk Low" lands in SB Dunk, not Dunk Low, and "Jordan 11" never counts as "Jordan 1".
To add a sneaker, add a line in the same format.

In [ ]:
import re

def norm(s):
    return re.sub(r'[^a-z0-9]+', ' ', str(s).lower()).strip()

# (class folder, brand, display name, regexes that must ALL match, regexes that must NOT match)
# Order matters: the first rule that matches wins, so specific models come before general ones.
CLASS_RULES = [
    ('nike_sb_dunk',        'Nike',        'Nike SB Dunk',            [r'\bsb\b.*\bdunk\b|\bdunk\b.*\bsb\b'], []),
    ('nike_dunk_low',       'Nike',        'Nike Dunk Low',           [r'\bdunk low\b'], []),
    ('nike_dunk_high',      'Nike',        'Nike Dunk High',          [r'\bdunk high\b'], []),
    ('air_jordan_1',        'Jordan',      'Air Jordan 1',            [r'\bjordan 1\b'], []),
    ('air_jordan_3',        'Jordan',      'Air Jordan 3',            [r'\bjordan 3\b'], []),
    ('air_jordan_4',        'Jordan',      'Air Jordan 4',            [r'\bjordan 4\b'], []),
    ('air_jordan_5',        'Jordan',      'Air Jordan 5',            [r'\bjordan 5\b'], []),
    ('air_jordan_6',        'Jordan',      'Air Jordan 6',            [r'\bjordan 6\b'], []),
    ('air_jordan_11',       'Jordan',      'Air Jordan 11',           [r'\bjordan 11\b'], []),
    ('air_jordan_12',       'Jordan',      'Air Jordan 12',           [r'\bjordan 12\b'], []),
    ('air_jordan_13',       'Jordan',      'Air Jordan 13',           [r'\bjordan 13\b'], []),
    ('nike_air_force_1',    'Nike',        'Nike Air Force 1',        [r'\bair force 1\b|\baf1\b'], []),
    ('nike_air_max_97',     'Nike',        'Nike Air Max 97',         [r'\bair max 97\b'], []),
    ('nike_air_max_95',     'Nike',        'Nike Air Max 95',         [r'\bair max 95\b'], []),
    ('nike_air_max_90',     'Nike',        'Nike Air Max 90',         [r'\bair max 90\b'], []),
    ('nike_air_max_270',    'Nike',        'Nike Air Max 270',        [r'\bair max 270\b'], []),
    ('nike_air_max_1',      'Nike',        'Nike Air Max 1',          [r'\bair max 1\b'], [r'\bair max 1 97\b']),
    ('nike_vapormax',       'Nike',        'Nike Air VaporMax',       [r'\bvapor ?max\b'], []),
    ('nike_blazer',         'Nike',        'Nike Blazer',             [r'\bblazer\b'], []),
    ('nike_cortez',         'Nike',        'Nike Cortez',             [r'\bcortez\b'], []),
    ('nike_air_huarache',   'Nike',        'Nike Air Huarache',       [r'\bhuarache\b'], []),
    ('yeezy_boost_350',     'Adidas',      'Adidas Yeezy Boost 350',  [r'\byeezy\b.*\b350\b|\b350 v2\b'], []),
    ('yeezy_500',           'Adidas',      'Adidas Yeezy 500',        [r'\byeezy\b.*\b500\b'], []),
    ('yeezy_700',           'Adidas',      'Adidas Yeezy 700',        [r'\byeezy\b.*\b700\b'], []),
    ('yeezy_foam_runner',   'Adidas',      'Adidas Yeezy Foam Runner', [r'\bfoam (runner|rnnr|rnr)\b'], []),
    ('yeezy_slide',         'Adidas',      'Adidas Yeezy Slide',      [r'\byeezy slide\b'], []),
    ('adidas_ultraboost',   'Adidas',      'Adidas Ultraboost',       [r'\bultra ?boost\b'], []),
    ('adidas_nmd',          'Adidas',      'Adidas NMD',              [r'\bnmd\b'], []),
    ('adidas_superstar',    'Adidas',      'Adidas Superstar',        [r'\bsuperstar\b'], []),
    ('adidas_stan_smith',   'Adidas',      'Adidas Stan Smith',       [r'\bstan smith\b'], []),
    ('adidas_samba',        'Adidas',      'Adidas Samba',            [r'\bsamba\b'], []),
    ('adidas_gazelle',      'Adidas',      'Adidas Gazelle',          [r'\bgazelle\b'], []),
    ('adidas_forum',        'Adidas',      'Adidas Forum',            [r'\badidas\b.*\bforum\b|\bforum (low|high|mid|84)\b'], []),
    ('new_balance_9060',    'New Balance', 'New Balance 9060',        [r'\b9060\b'], []),
    ('new_balance_2002r',   'New Balance', 'New Balance 2002R',       [r'\b2002 ?r\b'], []),
    ('new_balance_550',     'New Balance', 'New Balance 550',         [r'\bnew balance\b.*\b550\b|\bnb ?550\b|\bbb550'], []),
    ('new_balance_990',     'New Balance', 'New Balance 990',         [r'\bnew balance\b.*\b990(v\d)?\b|\bm990'], []),
    ('new_balance_992',     'New Balance', 'New Balance 992',         [r'\bnew balance\b.*\b992(v\d)?\b|\bm992'], []),
    ('new_balance_574',     'New Balance', 'New Balance 574',         [r'\bnew balance\b.*\b574(v\d)?\b'], []),
    ('new_balance_327',     'New Balance', 'New Balance 327',         [r'\bnew balance\b.*\b327(v\d)?\b'], []),
    ('asics_gel_kayano',    'ASICS',       'ASICS Gel-Kayano',        [r'\bkayano\b'], []),
    ('asics_gel_lyte_iii',  'ASICS',       'ASICS Gel-Lyte III',      [r'\bgel lyte (iii|3)\b'], []),
    ('asics_gel_1130',      'ASICS',       'ASICS Gel-1130',          [r'\bgel 1130\b'], []),
    ('converse_chuck_70',   'Converse',    'Converse Chuck 70',       [r'\bchuck 70\b|\bchuck taylor all star 70\b'], []),
    ('converse_chuck_taylor', 'Converse',  'Converse Chuck Taylor All Star', [r'\bconverse\b.*\b(chuck|all star)\b|\bchuck taylor\b'], [r'\b70\b']),
    ('vans_old_skool',      'Vans',        'Vans Old Skool',          [r'\bold skool\b'], []),
    ('vans_sk8_hi',         'Vans',        'Vans Sk8-Hi',             [r'\bsk8 hi\b'], []),
    ('vans_slip_on',        'Vans',        'Vans Slip-On',            [r'\bvans\b.*\bslip on\b'], []),
    ('puma_suede',          'Puma',        'Puma Suede',              [r'\bpuma\b.*\bsuede\b'], []),
    ('reebok_club_c',       'Reebok',      'Reebok Club C',           [r'\bclub c\b'], []),
]
_COMPILED = [(k, [re.compile(p) for p in inc], [re.compile(p) for p in exc]) for k, _, _, inc, exc in CLASS_RULES]

def classify(label_text):
    n = norm(label_text)
    for key, inc, exc in _COMPILED:
        if all(p.search(n) for p in inc) and not any(p.search(n) for p in exc):
            return key
    return None

print(len(CLASS_RULES), 'sneaker models defined')
for t in ['Air Jordan 1 Retro High Chicago', 'Nike SB Dunk Low Travis Scott', 'New Balance 990v5 Grey']:
    print(f'  example: {t!r} -> {classify(t)}')

## 3. Download the three datasets (10–20 minutes)

If Kaggle asks you to log in: kaggle.com → Settings → API → Create New Token, then add `KAGGLE_USERNAME` and
`KAGGLE_KEY` in Colab's **Secrets** (key icon on the left) with notebook access on. If one source fails, the
others still work, so you can keep going.

In [ ]:
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
except Exception:
    pass

import kagglehub
SOURCES = {}
for name, slug in [('popular', 'nikolasgegenava/sneakers-classification'),
                   ('stockx50k', 'sebastiaanjohn/sneakers')]:
    try:
        SOURCES[name] = Path(kagglehub.dataset_download(slug))
        print(f'{name}: downloaded to {SOURCES[name]}')
    except Exception as e:
        print(f'{name}: FAILED ({e}). Continuing without it.')

try:
    from huggingface_hub import snapshot_download
    SOURCES['hf93k'] = Path(snapshot_download('ipogorelov/sneakers', repo_type='dataset',
                                              allow_patterns=['*.parquet']))
    print('hf93k: downloaded to', SOURCES['hf93k'])
except Exception as e:
    print(f'hf93k: FAILED ({e}). Continuing without it.')

## 4. Sort every photo into a sneaker model

For the Hugging Face dataset, the notebook detects which column holds the image and which hold the text labels,
and prints an example row. **Check that the example label looks like a sneaker name.**

In [ ]:
IMG_EXTS = {'.jpg', '.jpeg', '.png', '.webp'}
cands = defaultdict(list)            # model -> list of photo references
per_source = defaultdict(Counter)    # source -> photos per model

def add_folder_source(tag, root):
    for d in root.rglob('*'):
        if not d.is_dir():
            continue
        cls = classify(d.name)
        if not cls:
            continue
        for f in d.iterdir():
            if f.suffix.lower() in IMG_EXTS:
                cands[cls].append(('file', str(f)))
                per_source[tag][cls] += 1

for tag in ['popular', 'stockx50k']:
    if tag in SOURCES:
        add_folder_source(tag, SOURCES[tag])
        print(f'{tag}: {sum(per_source[tag].values()):,} photos matched')

HF_FILES, HF_IMG_COL = [], None
if 'hf93k' in SOURCES:
    import pyarrow.parquet as pq
    HF_FILES = sorted(SOURCES['hf93k'].rglob('*.parquet'))
    first = pq.read_table(HF_FILES[0]).slice(0, 1).to_pylist()[0]
    is_img = lambda v: isinstance(v, (bytes, bytearray)) or (isinstance(v, dict) and 'bytes' in v)
    HF_IMG_COL = next((c for c, v in first.items() if is_img(v)), None)
    text_cols = [c for c, v in first.items() if isinstance(v, str)]
    print('\nHugging Face columns:', list(first.keys()))
    print('  image column:', HF_IMG_COL, '| label columns:', text_cols)
    print('  example label:', {c: first[c] for c in text_cols})
    if HF_IMG_COL is None or not text_cols:
        print('  Could not find image/label columns, so skipping this source.')
    else:
        for fi, f in enumerate(HF_FILES):
            for ri, row in enumerate(pq.read_table(f, columns=text_cols).to_pylist()):
                cls = classify(' '.join(str(row[c] or '') for c in text_cols))
                if cls:
                    cands[cls].append(('hf', fi, ri))
                    per_source['hf93k'][cls] += 1
        print(f'hf93k: {sum(per_source["hf93k"].values()):,} photos matched')

print(f'\n{"model":26s} {"popular":>8s} {"stockx50k":>10s} {"hf93k":>7s} {"total":>7s}')
for cls, *_ in CLASS_RULES:
    n = [per_source[s][cls] for s in ['popular', 'stockx50k', 'hf93k']]
    note = '' if sum(n) >= MIN_PER_CLASS else '  (too few, will be skipped)'
    print(f'{cls:26s} {n[0]:8,d} {n[1]:10,d} {n[2]:7,d} {sum(n):7,d}{note}')

## 5. Build the training set

Takes up to 600 photos per model from all sources combined, removes exact duplicates (the datasets overlap),
shrinks them, and saves them to Colab's local disk. About 10–15 minutes.

In [ ]:
from tqdm.auto import tqdm
shutil.rmtree(LOCAL_DATA, ignore_errors=True)
rng = random.Random(SEED)
seen = set()
kept = {}

def save_bytes(data, out):
    h = hashlib.md5(data).hexdigest()
    if h in seen:
        return False
    try:
        img = ImageOps.exif_transpose(Image.open(io.BytesIO(data))).convert('RGB')
        if min(img.size) < 64:
            return False  # skip tiny thumbnails
        img.thumbnail((IMG_SIDE, IMG_SIDE))
        out.parent.mkdir(parents=True, exist_ok=True)
        img.save(out, 'JPEG', quality=88)
        seen.add(h)
        return True
    except Exception:
        return False

# shuffle and oversample a little, since some photos turn out to be duplicates or broken
picked = {cls: rng.sample(refs, min(len(refs), int(MAX_PER_CLASS * 1.3)))
          for cls, refs in cands.items() if len(refs) >= MIN_PER_CLASS}
hf_needed = defaultdict(list)  # parquet file -> [(row, model)]
for cls, refs in tqdm(picked.items(), desc='folder photos'):
    kept[cls] = 0
    for ref in refs:
        if kept[cls] >= MAX_PER_CLASS:
            break
        if ref[0] == 'file':
            if save_bytes(Path(ref[1]).read_bytes(), LOCAL_DATA / cls / f'{kept[cls]:04d}.jpg'):
                kept[cls] += 1
        else:
            hf_needed[ref[1]].append((ref[2], cls))

if hf_needed:
    import pyarrow.parquet as pq
    for fi, wanted in tqdm(hf_needed.items(), desc='hugging face files'):
        col = pq.read_table(HF_FILES[fi], columns=[HF_IMG_COL]).column(0).to_pylist()
        for ri, cls in wanted:
            if kept[cls] >= MAX_PER_CLASS:
                continue
            v = col[ri]
            data = v['bytes'] if isinstance(v, dict) else v
            if data and save_bytes(data, LOCAL_DATA / cls / f'{kept[cls]:04d}.jpg'):
                kept[cls] += 1

for cls, n in list(kept.items()):
    if n < MIN_PER_CLASS:
        shutil.rmtree(LOCAL_DATA / cls, ignore_errors=True)
        del kept[cls]
print(f'{len(kept)} sneaker models, {sum(kept.values()):,} photos (after removing duplicates)\n')
for cls in sorted(kept):
    print(f'  {cls:26s} {kept[cls]:4d}')

In [ ]:
# one sample photo per model, as a quick sanity check
import matplotlib.pyplot as plt
classes = sorted(kept)
cols = 8
rows = int(np.ceil(len(classes) / cols))
fig, axes = plt.subplots(rows, cols, figsize=(2.2 * cols, 2.4 * rows))
for ax in np.array(axes).ravel():
    ax.axis('off')
for ax, cls in zip(np.array(axes).ravel(), classes):
    f = sorted((LOCAL_DATA / cls).glob('*.jpg'))[0]
    ax.imshow(Image.open(f)); ax.set_title(cls, fontsize=7)
plt.tight_layout(); plt.show()

## 6. Split and set up training (same method as notebook 2)

In [ ]:
from torchvision import datasets, transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset, DataLoader

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

base = datasets.ImageFolder(LOCAL_DATA)
class_names = base.classes
labels = np.array(base.targets)
idx = np.arange(len(labels))
train_idx, temp_idx = train_test_split(idx, test_size=0.30, stratify=labels, random_state=SEED)
val_idx, test_idx = train_test_split(temp_idx, test_size=0.50, stratify=labels[temp_idx], random_state=SEED)

train_ds = Subset(datasets.ImageFolder(LOCAL_DATA, transform=train_tf), train_idx)
val_ds   = Subset(datasets.ImageFolder(LOCAL_DATA, transform=eval_tf), val_idx)
test_ds  = Subset(datasets.ImageFolder(LOCAL_DATA, transform=eval_tf), test_idx)
BATCH = 64
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
print(f'{len(class_names)} classes | train {len(train_idx):,}, val {len(val_idx):,}, test {len(test_idx):,}')

In [ ]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, len(class_names))
model = model.to(device)

train_counts = np.bincount(labels[train_idx], minlength=len(class_names))
class_weights = torch.tensor(train_counts.sum() / (len(train_counts) * train_counts),
                             dtype=torch.float32, device=device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))  # mixed precision = faster on T4

print('Trainable parameters:', sum(p.numel() for p in model.parameters()))

## 7. Train (30–45 minutes)

Same two-phase method as before: 2 epochs training only the new last layer, then up to 12 epochs fine-tuning
everything. The best version is saved to Drive after every improvement, so a disconnect doesn't lose it.

In [ ]:
history = []
best_acc = 0.0
BEST_PATH = MODEL_DIR / 'best_weights_v2.pt'

def run_epoch(loader, train, optimizer=None):
    model.train(train)
    total_loss, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
                out = model(x)
                loss = criterion(out, y)
            if train:
                optimizer.zero_grad()
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            total_loss += loss.item() * x.size(0)
            correct += (out.argmax(1) == y).sum().item()
            n += x.size(0)
    return total_loss / n, correct / n

def train_phase(name, epochs, lr, freeze_backbone, patience=4):
    global best_acc
    for p in model.features.parameters():
        p.requires_grad = not freeze_backbone
    optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],
                                  lr=lr, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    bad_epochs = 0
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = run_epoch(train_loader, True, optimizer)
        va_loss, va_acc = run_epoch(val_loader, False)
        scheduler.step()
        history.append(dict(phase=name, train_loss=tr_loss, train_acc=tr_acc,
                            val_loss=va_loss, val_acc=va_acc))
        note = ''
        if va_acc > best_acc:
            best_acc, bad_epochs = va_acc, 0
            torch.save(model.state_dict(), BEST_PATH)
            note = '  <- best so far, saved'
        else:
            bad_epochs += 1
        print(f'[{name}] epoch {epoch:2d}/{epochs}  train acc {tr_acc:.3f}  '
              f'val acc {va_acc:.3f}  val loss {va_loss:.3f}  ({time.time() - t0:.0f}s){note}')
        if bad_epochs >= patience:
            print('Stopping early: validation accuracy stopped improving.')
            break

train_phase('head', epochs=2, lr=1e-3, freeze_backbone=True)
train_phase('finetune', epochs=12, lr=1e-4, freeze_backbone=False, patience=3)
print(f'\nBest validation accuracy: {best_acc:.3f}')

In [ ]:
import matplotlib.pyplot as plt

ep = range(1, len(history) + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ep, [h['train_loss'] for h in history], label='train')
axes[0].plot(ep, [h['val_loss'] for h in history], label='validation')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(ep, [h['train_acc'] for h in history], label='train')
axes[1].plot(ep, [h['val_acc'] for h in history], label='validation')
axes[1].set_title('Accuracy'); axes[1].set_xlabel('epoch'); axes[1].legend()
plt.tight_layout()
plt.savefig(MODEL_DIR / 'training_curves.png', dpi=150)
plt.show()

## 8. Evaluate on the test set

In [ ]:
from sklearn.metrics import classification_report, ConfusionMatrixDisplay

model.load_state_dict(torch.load(BEST_PATH, map_location=device))
model.eval()

all_probs, all_y = [], []
with torch.no_grad():
    for x, y in test_loader:
        all_probs.append(torch.softmax(model(x.to(device)), dim=1).cpu())
        all_y.append(y)
probs = torch.cat(all_probs).numpy()
y_true = torch.cat(all_y).numpy()
y_pred = probs.argmax(1)

top1 = float((y_pred == y_true).mean())
top3 = float(np.mean([t in np.argsort(p)[-3:] for p, t in zip(probs, y_true)]))
print(f'Test top-1 accuracy: {top1:.3f}')
print(f'Test top-3 accuracy: {top3:.3f}\n')
print(classification_report(y_true, y_pred, target_names=class_names, digits=3))

size = max(9, len(class_names) * 0.42)
fig, ax = plt.subplots(figsize=(size, size * 0.9))
ConfusionMatrixDisplay.from_predictions(y_true, y_pred, display_labels=class_names,
                                        xticks_rotation=90, cmap='Blues', ax=ax, colorbar=False,
                                        include_values=len(class_names) <= 20)
ax.set_title('Confusion matrix (test set)')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'confusion_matrix.png', dpi=150)
plt.show()

In [ ]:
conf = probs.max(1)
print(f'{"threshold":>9s} {"answered":>9s} {"accuracy when answered":>23s}')
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    mask = conf >= t
    acc = (y_pred[mask] == y_true[mask]).mean() if mask.any() else float('nan')
    print(f'{t:9.1f} {mask.mean():9.1%} {acc:23.1%}')

CONFIDENCE_THRESHOLD = 0.6  # change this after looking at the table

In [ ]:
from PIL import Image

wrong = np.where(y_pred != y_true)[0]
print(f'{len(wrong)} mistakes out of {len(y_true)} test images')

show = wrong[:12]
if len(show):
    cols = 4
    rows = int(np.ceil(len(show) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    for ax in np.array(axes).ravel():
        ax.axis('off')
    for ax, i in zip(np.array(axes).ravel(), show):
        path = base.samples[test_idx[i]][0]
        ax.imshow(Image.open(path).convert('RGB'))
        ax.set_title(f'true: {class_names[y_true[i]]}\npred: {class_names[y_pred[i]]} ({probs[i].max():.0%})',
                     fontsize=9)
    plt.tight_layout()
    plt.show()

## 9. Save the new model

Your first model is copied to `SoleSearch/models/v1_backup/` before it is replaced.

In [ ]:
# keep the first model as a backup
backup = MODEL_DIR / 'v1_backup'
backup.mkdir(exist_ok=True)
for f in ['solesearch_effnet_b0.pt', 'metrics.json', 'confusion_matrix.png', 'training_curves.png']:
    if (MODEL_DIR / f).exists() and not (backup / f).exists():
        shutil.copy(MODEL_DIR / f, backup / f)

package = {
    'arch': 'efficientnet_b0',
    'state_dict': model.state_dict(),
    'class_names': class_names,
    'img_size': IMG_SIZE,
    'mean': MEAN,
    'std': STD,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
}
torch.save(package, MODEL_DIR / 'solesearch_effnet_b0.pt')

metrics = {
    'test_top1_accuracy': top1,
    'test_top3_accuracy': top3,
    'best_val_accuracy': best_acc,
    'confidence_threshold': CONFIDENCE_THRESHOLD,
    'num_images': {'train': len(train_idx), 'val': len(val_idx), 'test': len(test_idx)},
    'class_names': class_names,
    'history': history,
}
(MODEL_DIR / 'metrics.json').write_text(json.dumps(metrics, indent=1))

size_mb = (MODEL_DIR / 'solesearch_effnet_b0.pt').stat().st_size / 1e6
print(f'Saved model ({size_mb:.0f} MB) and metrics to {MODEL_DIR}')

## 10. Pick a display photo for every sneaker

For each model, the photo the AI is most confident about becomes its display photo in the app.

In [ ]:
EXPORT = Path('/content/export')
shutil.rmtree(EXPORT, ignore_errors=True); EXPORT.mkdir()
(PROJECT / 'displays').mkdir(exist_ok=True)

def crop_4x3(img, size=(800, 600)):
    w, h = img.size; target = size[0] / size[1]
    if w / h > target:
        nw = int(h * target); left = (w - nw) // 2; img = img.crop((left, 0, left + nw, h))
    else:
        nh = int(w / target); top = (h - nh) // 2; img = img.crop((0, top, w, top + nh))
    return img.resize(size, Image.LANCZOS)

model.eval()
for ci, cls in enumerate(tqdm(class_names, desc='display photos')):
    files = sorted((LOCAL_DATA / cls).glob('*.jpg'))[:80]
    best, best_score = None, -1
    for f in files:
        img = Image.open(f).convert('RGB')
        with torch.no_grad():
            p = torch.softmax(model(eval_tf(img).unsqueeze(0).to(device)), 1)[0, ci].item()
        w, h = img.size
        score = p + (0.02 if 1.15 <= w / h <= 1.9 else 0)  # prefer side-view shaped photos
        if score > best_score:
            best, best_score = img, score
    out = crop_4x3(best)
    out.save(EXPORT / f'{cls}.jpg', 'JPEG', quality=86)
    out.save(PROJECT / 'displays' / f'{cls}.jpg', 'JPEG', quality=86)
print(len(list(EXPORT.glob('*.jpg'))), 'display photos saved')

## 11. Download everything the app needs

This downloads **`solesearch_update.zip`**. Unzip it and upload **all the files inside** to the top level of
your GitHub repo (they replace the old model, metrics and charts, and add the display photos).

In [ ]:
for f in ['solesearch_effnet_b0.pt', 'metrics.json', 'confusion_matrix.png', 'training_curves.png']:
    shutil.copy(MODEL_DIR / f, EXPORT / f)
shutil.make_archive('/content/solesearch_update', 'zip', EXPORT)
size = Path('/content/solesearch_update.zip').stat().st_size / 1e6
print(f'{len(list(EXPORT.iterdir()))} files, {size:.0f} MB')
from google.colab import files
files.download('/content/solesearch_update.zip')